<div dir=rtl style="text-align: right">

# שיעור: מסווג חתולים מול כלבים — עם CNN
## Convolutional Neural Network עם NumPy בלבד

**מה נלמד:**
- למה Fully Connected נכשל עם תמונות
- ההיסטוריה של CNN — מהמוח לAlexNet
- ארכיטקטורת CNN: Convolution, ReLU, MaxPool
- **גזירה מתמטית מלאה** של Backpropagation דרך Conv ו-MaxPool
- מימוש מ-scratch עם NumPy בלבד
- אימון על CIFAR-10 — חתולים מול כלבים

---

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import os, urllib.request, tarfile, pickle, time

# עוזר לכיתוב RTL בגרפים
def h(s):
    return s[::-1]

print('ספריות נטענו בהצלחה')

<div dir=rtl style="text-align: right">

---

## חלק א׳: מגבלות Fully Connected לתמונות

### הבעיה עם FC

בשיעור הקודם בנינו רשת Fully Connected שעבדה על **HOG features** — לא על הפיקסלים הגולמיים.  
למה? כי FC ישירות על פיקסלים סובל מכמה בעיות יסודיות:

### 1. פיצוץ פרמטרים

תמונת 32×32 RGB = **3,072 ערכים**.  
שכבת FC עם 512 נוירונים = 3,072 × 512 = **1,572,864 משקלים** רק בשכבה הראשונה.

תמונת 224×224 (סטנדרט) = 150,528 פיקסלים × 512 = **77 מיליון משקלים בשכבה אחת!**  

### 2. אין רגישות למיקום

FC "רואה" את הפיקסלים כוקטור שטוח — **סדר מרחבי אבד לחלוטין**.  
אוזן של חתול בפינה שמאלית == אוזן בפינה ימינית? עבור FC, אלה ערכים שונים לגמרי!

### 3. אין שיתוף משקלים

אם הרשת למדה לזהות קצה אופקי בפינה אחת של התמונה — היא **לא יודעת** שזה אותו קצה בפינה אחרת.  
כל מיקום לומד מאפרוע, בנפרד, מאפס.

### 4. לא אינוריאנטי לטרנספורמציות

אם החתול זזה 2 פיקסלים שמאלה — הרשת FC "לא מכירה" אותו יותר.

---

### למה CNN פותר את זה?

| בעיה | פתרון ב-CNN |
|------|-------------|
| פיצוץ פרמטרים | **שיתוף משקלים** — פילטר אחד רץ על כל התמונה |
| איבוד מרחביות | **שימור מבנה 2D** — הקונבולוציה שומרת על גאוגרפיה |
| חוסר אינווריאנטיות | **MaxPool** — מגדיל חסינות להזזות קטנות |
| אין היררכיה | **שכבות עמוקות** — שכבות ראשונות: קצוות, שכבות עמוקות: עיניים/אוזניים |

</div>

<div dir=rtl style="text-align: right">

---

## חלק ב׳: ההיסטוריה של CNN — מהמוח לAlexNet

### 1959 — Hubel & Wiesel: קורטקס הראייה של החתול

**David Hubel** ו-**Torsten Wiesel** (זוכי נובל 1981) ביצעו ניסוי מכונן:  
הם תקעו אלקטרודות בקורטקס הראייה של חתולים ומצאו שלא כל הנוירונים מגיבים לכל גירוי.

- **Simple cells** — מגיבות לקווים בזוויות ספציפיות, **בחלק ספציפי של שדה הראייה**
- **Complex cells** — מגיבות לאותה זווית אבל **ללא תלות במיקום** (אינווריאנטיות מקומית)
- **Hypercomplex cells** — מגיבות לפינות ומכלולים מורכבים יותר

המסקנה המפתיעה: **הראייה היא היררכית ומקומית**. הקורטקס לא עובד כמו FC!

---

### 1980 — Fukushima: Neocognitron

**Kunihiko Fukushima** בנה את **Neocognitron** — רשת שמדמה את Simple/Complex cells של Hubel-Wiesel.  
הרשת כללה שכבות חישוביות מקומיות ושכבות "פולינג" (לסיכום מקומי).  
**לא אומנה בגרדיאנטים** — הלמידה הייתה unsupervised.

---

### 1989 — LeCun: LeNet ו-Backprop דרך Conv

**Yann LeCun** לקח את הרעיון של Fukushima וחיבר אותו עם **Backpropagation**.  
הפריצה: גזר (ידנית) כיצד הגרדיאנט עובר דרך שכבת קונבולוציה.

הוא בנה **LeNet-5** לזיהוי ספרות ידניות (MNIST) — הבנק הפדרלי האמריקאי השתמש בה לקריאת שיקים.  
**LeNet-5 היא הסבא-רבא של כל CNN מודרני.**

ארכיטקטורה: `Conv → Pool → Conv → Pool → FC → FC → Output`

---

### 2012 — AlexNet: המהפכה

בתחרות **ImageNet 2012**, **Alex Krizhevsky** (מקבוצת Geoffrey Hinton) הכניס CNN ל-GPU.  
**AlexNet** השיגה error rate של 15.3% לעומת 26.2% של הגישות הקלאסיות — **פי 2 טוב יותר!**

חידושים שלא היו ב-LeNet:
- **ReLU** (במקום Tanh) — אימון מהיר פי 6
- **Dropout** — מניעת overfitting
- **Data Augmentation** — הגדלת dataset מלאכותית
- **GPU Training** — 5 ימי אימון במקום שבועות

מאז 2012: **Deep Learning השתלט על Computer Vision.**

---

### ציר הזמן:

```
1959: Hubel-Wiesel → קורטקס ראייה היררכי
1980: Fukushima   → Neocognitron (ללא gradient)
1989: LeCun       → LeNet (conv + backprop!)
2012: AlexNet     → GPU + ReLU + Dropout → מהפכה
2014: VGG, GoogLeNet → עמוק יותר ויותר
2015: ResNet      → 152 שכבות, skip connections
2017: Transformer → CNN מתחיל לוותר לטרנספורמרים
```

</div>

<div dir=rtl style="text-align: right">

---

## חלק ג׳: ארכיטקטורת CNN — Forward Pass

### מהי קונבולוציה?

פילטר (kernel) בגודל $K \times K$ "רץ" על התמונה ומחשב מכפלה עם כל אזור:

$$Z[f, i, j] = \sum_{c=0}^{C_{in}-1} \sum_{m=0}^{K-1} \sum_{n=0}^{K-1} W[f, c, m, n] \cdot X[c,\; i + m,\; j + n] + b[f]$$

סימונים:
- $X$: קלט — $(C_{in}, H, W)$
- $W$: פילטרים — $(C_{out}, C_{in}, K, K)$
- $b$: bias — $(C_{out},)$
- $Z$: פלט — $(C_{out}, H_{out}, W_{out})$
- $H_{out} = H - K + 1$ (ללא padding), עם padding=1: $H_{out} = H$

### ארכיטקטורת הרשת שנבנה:

```
Input:  (N, 3, 32, 32)         — batch של תמונות RGB
Conv1:  3 → 32 פילטרים, 3×3  — (N, 32, 32, 32)
ReLU:                           — (N, 32, 32, 32)
MaxPool 2×2:                    — (N, 32, 16, 16)
Conv2:  32 → 64 פילטרים, 3×3 — (N, 64, 16, 16)
ReLU:                           — (N, 64, 16, 16)
MaxPool 2×2:                    — (N, 64, 8, 8)
Flatten:                        — (N, 4096)
FC1:    4096 → 256             — (N, 256)
ReLU + Dropout(0.5):            — (N, 256)
FC2:    256 → 1                — (N, 1)
Sigmoid:                        — סיכוי לכלב
```

### MaxPool

לוקח את **המקסימום** בכל חלון 2×2, עם stride=2:

$$Z[c, i, j] = \max_{0 \le m,n < K} X[c,\; 2i + m,\; 2j + n]$$

מקטין רזולוציה פי 2 בכל ממד → חסינות לטרנספורמציות קטנות.

</div>

<div dir=rtl style="text-align: right">

---

## חלק ד׳: גזירה מתמטית — Backprop דרך Convolution

זה החלק הכי מעמיק בשיעור. נגזור את כל הגרדיאנטים מאפס.

---

### הגדרות

נעבוד לפשטות על batch יחיד $(N=1)$ וערוץ יחיד $(C_{in}=C_{out}=1)$.

Forward pass:
$$Z[i,j] = \sum_{m=0}^{K-1} \sum_{n=0}^{K-1} W[m,n] \cdot X[i+m,\, j+n] + b$$

נניח שיש לנו $\frac{\partial L}{\partial Z[i,j]}$ — הגרדיאנט של הloss ביחס לכל פלט. נסמן אותו $\delta[i,j]$.

---

### גרדיאנט ביחס ל-bias: $\frac{\partial L}{\partial b}$

כל פלט $Z[i,j]$ מושפע מ-$b$:
$$\frac{\partial Z[i,j]}{\partial b} = 1$$

לכן על פי chain rule:
$$\boxed{\frac{\partial L}{\partial b} = \sum_{i,j} \delta[i,j]}$$

**פרשנות:** מחברים את כל הגרדיאנטים מהפלט.

---

### גרדיאנט ביחס למשקל: $\frac{\partial L}{\partial W[m,n]}$

המשקל $W[m,n]$ משפיע על כל $Z[i,j]$:
$$\frac{\partial Z[i,j]}{\partial W[m,n]} = X[i+m,\, j+n]$$

על פי chain rule:
$$\boxed{\frac{\partial L}{\partial W[m,n]} = \sum_{i,j} \delta[i,j] \cdot X[i+m,\, j+n]}$$

**זוהי cross-correlation של $X$ עם $\delta$!**

בגרסה הכללית עם $C_{in}$ ערוצים ו-$C_{out}$ פילטרים:
$$\frac{\partial L}{\partial W[f,c,m,n]} = \sum_{i,j} \delta_f[i,j] \cdot X[c,\, i+m,\, j+n]$$

---

### גרדיאנט ביחס לקלט: $\frac{\partial L}{\partial X[p,q]}$

זה החלק הטריקי ביותר.

הפיקסל $X[p,q]$ משתתף בחישוב של $Z[i,j]$ כאשר $m = p-i$ ו-$n = q-j$:
$$\frac{\partial Z[i,j]}{\partial X[p,q]} = W[p-i,\, q-j]$$

(רק כאשר $0 \le p-i \le K-1$ ו-$0 \le q-j \le K-1$, כלומר $(i,j) \in [p-K+1, p] \times [q-K+1, q]$)

על פי chain rule:
$$\frac{\partial L}{\partial X[p,q]} = \sum_{i,j} \delta[i,j] \cdot W[p-i,\, q-j]$$

**זוהי קונבולוציה מלאה (full convolution) של $\delta$ עם $W$ הפוך!**

הPaddingשנחוץ: אם $Z$ הוא `valid` (ללא padding), אז הגרדיאנט לX מחייב padding=$K-1$ על $\delta$.

בגרסה הכללית:
$$\frac{\partial L}{\partial X[c,p,q]} = \sum_{f,m,n} \delta_f\left[p-m,\, q-n\right] \cdot W[f,c,m,n]$$

**ויזואלית:** הגרדיאנט "מתפשט" חזרה דרך הפילטר בכיוון ההפוך.

---

### פישוט עם im2col

קונבולוציה ניתן לכתוב כ**כפל מטריצות**. להלן ההמרה:

1. **im2col(X)**: כל חלון $K \times K$ (של כל הערוצים) הופך לשורה.  
   תוצאה: $X_{col}$ בגודל $(H_{out} \cdot W_{out},\; C_{in} \cdot K^2)$

2. **W_mat**: הפילטרים משוטחים לשורות.  
   תוצאה: $W_{mat}$ בגודל $(C_{out},\; C_{in} \cdot K^2)$

3. **Forward**: $Z_{col} = X_{col} \cdot W_{mat}^T + b$ — *כפל מטריצות!*

4. **Backward**:
   - $\nabla W_{mat} = \delta_{col}^T \cdot X_{col}$ — גרדיאנט המשקלים
   - $\nabla X_{col} = \delta_{col} \cdot W_{mat}$ — גרדיאנט הקלט
   - `col2im`($\nabla X_{col}$) — המרה חזרה לפורמט תמונה, עם **צבירת חפיפות**

שים לב: `col2im` **מחבר** גרדיאנטים חופפים — זה בדיוק מה שהגזירה דרשה (כל פיקסל תורם לכמה חלונות).

</div>

<div dir=rtl style="text-align: right">

---

## חלק ה׳: גזירה — Backprop דרך MaxPool

### Forward:

$$Z[c, i, j] = \max_{0 \le m,n < K} X[c,\; Ki+m,\; Kj+n]$$

נסמן את המיקום שמגיע למקסימום: $(m^*, n^*) = \underset{m,n}{\arg\max}\; X[c, Ki+m, Kj+n]$

### Backward:

הגרדיאנט עובר רק דרך **הפיקסל שהיה המקסימום**:

$$\frac{\partial Z[c,i,j]}{\partial X[c, p, q]} = \begin{cases} 1 & \text{אם } (p,q) = (Ki+m^*, Kj+n^*) \\ 0 & \text{אחרת} \end{cases}$$

לכן:
$$\frac{\partial L}{\partial X[c, Ki+m^*, Kj+n^*]} \mathrel{+}= \frac{\partial L}{\partial Z[c,i,j]}$$

**מימוש:** שמרים את מיקום המקסימום בforward, ובbackward מניחים את הגרדיאנט במיקום ההוא.

---

### ReLU — טריוויאלי:

$$\text{ReLU}(x) = \max(0, x)$$

$$\frac{\partial \text{ReLU}(x)}{\partial x} = \begin{cases} 1 & x > 0 \\ 0 & x \le 0 \end{cases}$$

### Sigmoid + BCE — גרדיאנט פשוט:

נסמן $a = \sigma(z) = \frac{1}{1+e^{-z}}$ ו-$L = -y\log a - (1-y)\log(1-a)$.

$$\frac{\partial L}{\partial z} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} = \left(-\frac{y}{a} + \frac{1-y}{1-a}\right) \cdot a(1-a) = a - y$$

$$\boxed{\frac{\partial L}{\partial z} = \frac{a - y}{N}}$$

אלגנטי! הגרדיאנט הוא פשוט ההפרש בין הניבוי לאמת.

</div>

In [ ]:
# ===== הורדת ועיבוד נתוני CIFAR-10 =====

def download_and_load_cifar10(data_dir='cifar10_data'):
    url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
    os.makedirs(data_dir, exist_ok=True)
    tar_path = os.path.join(data_dir, 'cifar-10-python.tar.gz')

    if not os.path.exists(os.path.join(data_dir, 'cifar-10-batches-py')):
        print('מוריד CIFAR-10...')
        urllib.request.urlretrieve(url, tar_path)
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(data_dir)
        print('הורדה הושלמה!')

    batch_dir = os.path.join(data_dir, 'cifar-10-batches-py')
    X_all, y_all = [], []
    for i in range(1, 6):
        with open(os.path.join(batch_dir, f'data_batch_{i}'), 'rb') as f:
            d = pickle.load(f, encoding='bytes')
        X_all.append(d[b'data'])
        y_all.append(np.array(d[b'labels']))
    X_train_full = np.concatenate(X_all).reshape(-1, 3, 32, 32)  # NCHW
    y_train_full = np.concatenate(y_all)

    with open(os.path.join(batch_dir, 'test_batch'), 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    X_test_full = np.array(d[b'data']).reshape(-1, 3, 32, 32)  # NCHW
    y_test_full = np.array(d[b'labels'])

    return X_train_full, y_train_full, X_test_full, y_test_full


CAT_IDX = 3
DOG_IDX = 5

def filter_cats_dogs(X, y):
    mask = (y == CAT_IDX) | (y == DOG_IDX)
    Xf = X[mask].astype(np.float32) / 255.0  # normalize to [0,1]
    yf = (y[mask] == DOG_IDX).astype(np.float32)  # 0=cat, 1=dog
    return Xf, yf


X_train_raw, y_train_raw, X_test_raw, y_test_raw = download_and_load_cifar10()
X_train, y_train = filter_cats_dogs(X_train_raw, y_train_raw)
X_test,  y_test  = filter_cats_dogs(X_test_raw,  y_test_raw)

# נרמול per-channel (mean/std על train)
mean = X_train.mean(axis=(0, 2, 3), keepdims=True)
std  = X_train.std(axis=(0, 2, 3), keepdims=True) + 1e-8
X_train = (X_train - mean) / std
X_test  = (X_test  - mean) / std

print(f'Train: {X_train.shape}, {y_train.shape} | חתולים: {(y_train==0).sum()}, כלבים: {(y_train==1).sum()}')
print(f'Test:  {X_test.shape},  {y_test.shape}  | חתולים: {(y_test==0).sum()},  כלבים: {(y_test==1).sum()}')
print(f'פורמט: NCHW — (batch, channels, height, width)')

In [ ]:
# הצגת דוגמאות
# שחזור למראה (0,1) לתצוגה
X_vis_train = (X_train * std.squeeze()[None,:,None,None] + mean.squeeze()[None,:,None,None])
X_vis_test  = (X_test  * std.squeeze()[None,:,None,None] + mean.squeeze()[None,:,None,None])
X_vis_train = np.clip(X_vis_train.transpose(0, 2, 3, 1), 0, 1)  # NHWC
X_vis_test  = np.clip(X_vis_test.transpose(0, 2, 3, 1), 0, 1)

cats = np.where(y_train == 0)[0][:8]
dogs = np.where(y_train == 1)[0][:8]

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle(h('חתולים (שורה 1) | כלבים (שורה 2)'), fontsize=14)
for j, idx in enumerate(cats):
    axes[0, j].imshow(X_vis_train[idx])
    axes[0, j].axis('off')
for j, idx in enumerate(dogs):
    axes[1, j].imshow(X_vis_train[idx])
    axes[1, j].axis('off')
plt.tight_layout()
plt.show()

<div dir=rtl style="text-align: right">

---

## חלק ו׳: מימוש NumPy — im2col ו-col2im

**im2col** ממיר batch של תמונות למטריצה שבה כל שורה היא חלון (patch) אחד.  
זה מאפשר לממש קונבולוציה כ**כפל מטריצות יחיד** — הרבה יותר מהיר מלולאות.

</div>

In [ ]:
# ===== im2col / col2im =====

def im2col(X, kh, kw, stride=1, pad=0):
    """
    ממיר batch של תמונות למטריצת patches.
    X: (N, C, H, W)
    מחזיר: col (N*H_out*W_out, C*kh*kw), H_out, W_out
    """
    N, C, H, W = X.shape
    if pad > 0:
        X = np.pad(X, ((0,0),(0,0),(pad,pad),(pad,pad)), mode='constant')
    _, _, H_pad, W_pad = X.shape
    H_out = (H_pad - kh) // stride + 1
    W_out = (W_pad - kw) // stride + 1

    # col[n, c, m, n_k, i, j] = X[n, c, i*stride+m, j*stride+n_k]
    col = np.zeros((N, C, kh, kw, H_out, W_out), dtype=X.dtype)
    for y in range(kh):
        for x in range(kw):
            col[:, :, y, x, :, :] = X[:, :,
                                       y : y + stride*H_out : stride,
                                       x : x + stride*W_out : stride]

    # (N, C, kh, kw, H_out, W_out) → (N*H_out*W_out, C*kh*kw)
    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N * H_out * W_out, -1)
    return col, H_out, W_out


def col2im(col, X_shape, kh, kw, stride=1, pad=0):
    """
    הפוך של im2col — צובר גרדיאנטים חופפים.
    col: (N*H_out*W_out, C*kh*kw)
    X_shape: (N, C, H, W) — צורת הקלט המקורי
    מחזיר: (N, C, H, W)
    """
    N, C, H, W = X_shape
    H_pad = H + 2*pad
    W_pad = W + 2*pad
    H_out = (H_pad - kh) // stride + 1
    W_out = (W_pad - kw) // stride + 1

    # (N*H_out*W_out, C*kh*kw) → (N, H_out, W_out, C, kh, kw)
    col_r = col.reshape(N, H_out, W_out, C, kh, kw).transpose(0, 3, 4, 5, 1, 2)

    X_padded = np.zeros((N, C, H_pad, W_pad), dtype=col.dtype)
    for y in range(kh):
        for x in range(kw):
            X_padded[:, :,
                     y : y + stride*H_out : stride,
                     x : x + stride*W_out : stride] += col_r[:, :, y, x, :, :]

    if pad > 0:
        return X_padded[:, :, pad:pad+H, pad:pad+W]
    return X_padded


# ===== בדיקת תקינות im2col/col2im =====
np.random.seed(42)
X_t = np.random.randn(2, 3, 8, 8).astype(np.float32)
col_t, ho, wo = im2col(X_t, 3, 3, stride=1, pad=1)
print(f'im2col: X={X_t.shape} → col={col_t.shape}  (H_out={ho}, W_out={wo})')
dX_t = col2im(col_t, X_t.shape, 3, 3, stride=1, pad=1)
print(f'col2im: col={col_t.shape} → {dX_t.shape}')
print('col2im לא הפיך מדויק (חיבור חפיפות) — OK לgradient!') 

In [ ]:
# ===== שכבת קונבולוציה =====

class ConvLayer:
    """
    שכבת קונבולוציה עם Adam.
    Forward:  Z_col = X_col @ W_mat.T + b
    Backward: dW = dZ_col.T @ X_col  |  dX_col = dZ_col @ W_mat
    """
    def __init__(self, in_ch, out_ch, k, stride=1, pad=1):
        self.in_ch, self.out_ch = in_ch, out_ch
        self.k, self.stride, self.pad = k, stride, pad

        # He initialization (מומלץ עם ReLU)
        fan_in = in_ch * k * k
        self.W = np.random.randn(out_ch, in_ch, k, k).astype(np.float32) * np.sqrt(2.0 / fan_in)
        self.b = np.zeros(out_ch, dtype=np.float32)

        # Adam moments
        self.mW = np.zeros_like(self.W); self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b); self.vb = np.zeros_like(self.b)
        self.t = 0

    def forward(self, X):
        self.X = X  # cache for backward
        N = X.shape[0]

        # im2col: (N*H_out*W_out, in_ch*k*k)
        self.X_col, self.H_out, self.W_out = im2col(X, self.k, self.k, self.stride, self.pad)

        # W_mat: (out_ch, in_ch*k*k)
        W_mat = self.W.reshape(self.out_ch, -1)

        # Z_col: (N*H_out*W_out, out_ch)
        Z_col = self.X_col @ W_mat.T + self.b

        # Reshape to (N, out_ch, H_out, W_out)
        Z = Z_col.reshape(N, self.H_out, self.W_out, self.out_ch).transpose(0, 3, 1, 2)
        return Z

    def backward(self, dZ):
        """
        dZ: (N, out_ch, H_out, W_out)
        מחזיר: dX (N, in_ch, H, W)
        """
        N = dZ.shape[0]
        W_mat = self.W.reshape(self.out_ch, -1)

        # (N, out_ch, H_out, W_out) → (N*H_out*W_out, out_ch)
        dZ_col = dZ.transpose(0, 2, 3, 1).reshape(-1, self.out_ch)

        # dW = dZ_col.T @ X_col    →   (out_ch, in_ch*k*k)
        self.dW = (dZ_col.T @ self.X_col).reshape(self.W.shape)
        self.db = dZ_col.sum(axis=0)

        # dX_col = dZ_col @ W_mat   →   (N*H_out*W_out, in_ch*k*k)
        dX_col = dZ_col @ W_mat

        # col2im: (N, in_ch, H, W)
        dX = col2im(dX_col, self.X.shape, self.k, self.k, self.stride, self.pad)
        return dX

    def update(self, lr, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        for param, grad, m, v in [
            (self.W, self.dW, self.mW, self.vW),
            (self.b, self.db, self.mb, self.vb)
        ]:
            m[:] = b1*m + (1-b1)*grad
            v[:] = b2*v + (1-b2)*grad**2
            param -= lr * (m/(1-b1**self.t)) / (np.sqrt(v/(1-b2**self.t)) + eps)


print('ConvLayer הוגדר')

In [ ]:
# ===== MaxPool, ReLU, Dropout, Flatten, FC =====

class MaxPoolLayer:
    """MaxPool 2D עם im2col."""
    def __init__(self, pool=2, stride=2):
        self.pool = pool
        self.stride = stride

    def forward(self, X):
        N, C, H, W = X.shape
        kh = kw = self.pool
        H_out = (H - kh) // self.stride + 1
        W_out = (W - kw) // self.stride + 1
        self.X_shape = X.shape
        self.H_out, self.W_out = H_out, W_out

        # ממיר כל ערוץ בנפרד
        X_r = X.reshape(N*C, 1, H, W)
        X_col, _, _ = im2col(X_r, kh, kw, self.stride, pad=0)
        # X_col: (N*C*H_out*W_out, kh*kw)

        self.max_idx = np.argmax(X_col, axis=1)  # שמירת מיקום המקסימום
        out = X_col[np.arange(len(X_col)), self.max_idx]
        self.X_col = X_col
        return out.reshape(N, C, H_out, W_out)

    def backward(self, dout):
        N, C, H, W = self.X_shape
        kh = kw = self.pool

        dX_col = np.zeros_like(self.X_col)
        # הגרדיאנט הולך רק לפיקסל המקסימלי
        dX_col[np.arange(len(dX_col)), self.max_idx] = dout.reshape(-1)

        dX_r = col2im(dX_col, (N*C, 1, H, W), kh, kw, self.stride, pad=0)
        return dX_r.reshape(N, C, H, W)


class ReLU:
    def forward(self, X):
        self.mask = X > 0
        return X * self.mask

    def backward(self, dout):
        return dout * self.mask


class Dropout:
    def __init__(self, rate=0.5):
        self.rate = rate
        self.training = True

    def forward(self, X):
        if self.training:
            self.mask = (np.random.rand(*X.shape) > self.rate).astype(np.float32) / (1 - self.rate)
            return X * self.mask
        return X

    def backward(self, dout):
        return dout * self.mask


class FlattenLayer:
    def forward(self, X):
        self.X_shape = X.shape
        return X.reshape(X.shape[0], -1)

    def backward(self, dout):
        return dout.reshape(self.X_shape)


class FCLayer:
    def __init__(self, in_size, out_size):
        self.W = np.random.randn(in_size, out_size).astype(np.float32) * np.sqrt(2.0 / in_size)
        self.b = np.zeros(out_size, dtype=np.float32)
        self.mW = np.zeros_like(self.W); self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b); self.vb = np.zeros_like(self.b)
        self.t = 0

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, dout):
        self.dW = self.X.T @ dout
        self.db = dout.sum(axis=0)
        return dout @ self.W.T

    def update(self, lr, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        for param, grad, m, v in [
            (self.W, self.dW, self.mW, self.vW),
            (self.b, self.db, self.mb, self.vb)
        ]:
            m[:] = b1*m + (1-b1)*grad
            v[:] = b2*v + (1-b2)*grad**2
            param -= lr * (m/(1-b1**self.t)) / (np.sqrt(v/(1-b2**self.t)) + eps)


print('כל השכבות הוגדרו: MaxPool, ReLU, Dropout, Flatten, FC')

In [ ]:
# ===== מודל CNN =====

class CNN:
    """
    ארכיטקטורה:
      Conv(3→32, 3×3, pad=1) → ReLU → MaxPool(2×2)
      Conv(32→64, 3×3, pad=1) → ReLU → MaxPool(2×2)
      Flatten → FC(4096→256) → ReLU → Dropout(0.5) → FC(256→1) → Sigmoid
    """
    def __init__(self):
        np.random.seed(42)
        self.conv1   = ConvLayer(3,  32, 3, pad=1)
        self.relu1   = ReLU()
        self.pool1   = MaxPoolLayer(2, 2)       # 32×32 → 16×16

        self.conv2   = ConvLayer(32, 64, 3, pad=1)
        self.relu2   = ReLU()
        self.pool2   = MaxPoolLayer(2, 2)       # 16×16 → 8×8

        self.flat    = FlattenLayer()           # 64*8*8 = 4096
        self.fc1     = FCLayer(4096, 256)
        self.relu3   = ReLU()
        self.drop    = Dropout(0.5)
        self.fc2     = FCLayer(256, 1)

        self.training = True

    def forward(self, X):
        self.drop.training = self.training
        z = self.conv1.forward(X)
        z = self.relu1.forward(z)
        z = self.pool1.forward(z)
        z = self.conv2.forward(z)
        z = self.relu2.forward(z)
        z = self.pool2.forward(z)
        z = self.flat.forward(z)
        z = self.fc1.forward(z)
        z = self.relu3.forward(z)
        z = self.drop.forward(z)
        z = self.fc2.forward(z)
        # Sigmoid — bce gradient: a - y (מחושב ב-backward)
        self.a = 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))
        return self.a

    def loss(self, y):
        eps = 1e-9
        y = y.reshape(-1, 1)
        return float(-np.mean(y * np.log(self.a + eps) + (1-y) * np.log(1-self.a + eps)))

    def backward(self, y):
        N = len(y)
        da = (self.a - y.reshape(-1, 1)) / N  # dL/dz_sigmoid (a-y)/N
        da = self.fc2.backward(da)
        da = self.drop.backward(da)
        da = self.relu3.backward(da)
        da = self.fc1.backward(da)
        da = self.flat.backward(da)
        da = self.pool2.backward(da)
        da = self.relu2.backward(da)
        da = self.conv2.backward(da)
        da = self.pool1.backward(da)
        da = self.relu1.backward(da)
        da = self.conv1.backward(da)

    def update(self, lr):
        self.conv1.update(lr)
        self.conv2.update(lr)
        self.fc1.update(lr)
        self.fc2.update(lr)

    def predict(self, X, batch=128):
        proba = self.predict_proba(X, batch)
        return (proba > 0.5).astype(int)

    def predict_proba(self, X, batch=128):
        self.training = False
        preds = []
        for i in range(0, len(X), batch):
            preds.append(self.forward(X[i:i+batch]).squeeze())
        self.training = True
        return np.concatenate(preds)


# בדיקת forward pass
model = CNN()
xb = X_train[:4]
model.training = False
out = model.forward(xb)
print(f'Forward OK: input={xb.shape} → output={out.shape}')
print(f'ערכים: {out.squeeze()}')

In [ ]:
# ===== לולאת אימון =====

def train_cnn(model, X_tr, y_tr, X_val, y_val,
              epochs=25, batch=64, lr=1e-3):

    N = len(X_tr)
    history = {'train_loss': [], 'val_loss': [],
               'train_acc': [],  'val_acc': []}
    best_acc = 0.0

    for ep in range(1, epochs+1):
        t0 = time.time()
        idx = np.random.permutation(N)
        X_tr, y_tr = X_tr[idx], y_tr[idx]

        ep_loss = 0.0
        model.training = True
        for i in range(0, N, batch):
            xb = X_tr[i:i+batch]
            yb = y_tr[i:i+batch]
            model.forward(xb)
            ep_loss += model.loss(yb)
            model.backward(yb)
            model.update(lr)

        n_batches = N // batch
        ep_loss /= n_batches

        # הערכה על subset מהtrain + כל ה-val
        tr_pred = model.predict(X_tr[:800])
        tr_acc  = np.mean(tr_pred == y_tr[:800])

        val_pred = model.predict(X_val)
        val_proba = model.predict_proba(X_val)
        val_acc  = np.mean(val_pred == y_val)

        eps_val = 1e-9
        val_loss = float(-np.mean(
            y_val * np.log(val_proba + eps_val) +
            (1-y_val) * np.log(1-val_proba + eps_val)))

        history['train_loss'].append(ep_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(float(tr_acc))
        history['val_acc'].append(float(val_acc))

        if val_acc > best_acc:
            best_acc = val_acc
            best_ep = ep

        print(f'Epoch {ep:2d}/{epochs} | loss={ep_loss:.4f} val_loss={val_loss:.4f} '
              f'| tr_acc={tr_acc:.3f} val_acc={val_acc:.3f} '
              f'| {time.time()-t0:.1f}s')

    print(f'\nהטוב ביותר: epoch {best_ep} | val_acc={best_acc:.3f}')
    return history


# אימון!
# ⚠️ על CPU זה יקח ~15-30 דקות (25 אפוקות)
# ב-Colab עם GPU הרבה יותר מהיר
print('מתחיל אימון...')
print('(ב-Colab CPU: ~15-30 דקות | ב-GPU: ~2-5 דקות)')
history = train_cnn(model, X_train.copy(), y_train.copy(), X_test, y_test,
                    epochs=25, batch=64, lr=1e-3)

In [ ]:
# ===== גרפי Loss ו-Accuracy =====

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep_range, history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(ep_range, history['val_loss'],   label='Val Loss',   color='tomato', linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title(h('אימון vs. ולידציה — Loss'))
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_range, history['train_acc'], label='Train Acc', color='steelblue')
axes[1].plot(ep_range, history['val_acc'],   label='Val Acc',   color='tomato', linestyle='--')
axes[1].axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='Chance')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title(h('אימון vs. ולידציה — Accuracy'))
axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.4, 1.0)

plt.tight_layout()
plt.show()
print(f'Val Accuracy הסופי: {history["val_acc"][-1]:.3f}')

In [ ]:
# ===== Confusion Matrix + Precision/Recall/F1 =====

y_pred = model.predict(X_test)
y_true = y_test.astype(int)

TP = int(np.sum((y_pred == 1) & (y_true == 1)))
TN = int(np.sum((y_pred == 0) & (y_true == 0)))
FP = int(np.sum((y_pred == 1) & (y_true == 0)))
FN = int(np.sum((y_pred == 0) & (y_true == 1)))

acc  = (TP + TN) / len(y_true)
prec = TP / (TP + FP + 1e-9)
rec  = TP / (TP + FN + 1e-9)
f1   = 2 * prec * rec / (prec + rec + 1e-9)

cm = np.array([[TN, FP], [FN, TP]])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['חתול (0)', 'כלב (1)'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['חתול (0)', 'כלב (1)'])
axes[0].set_xlabel('ניבוי'); axes[0].set_ylabel('אמת')
axes[0].set_title('Confusion Matrix')
for r in range(2):
    for c in range(2):
        axes[0].text(c, r, str(cm[r, c]),
                     ha='center', va='center',
                     color='white' if cm[r, c] > cm.max()*0.5 else 'black',
                     fontsize=16)

# מדדים
metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}
bars = axes[1].bar(metrics.keys(), metrics.values(),
                   color=['steelblue', 'coral', 'mediumseagreen', 'mediumpurple'])
axes[1].set_ylim(0, 1.1)
axes[1].set_title(h('מדדי ביצועים'))
for bar, v in zip(bars, metrics.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Accuracy:  {acc:.3f}  ({acc*100:.1f}%)')
print(f'Precision: {prec:.3f}')
print(f'Recall:    {rec:.3f}')
print(f'F1 Score:  {f1:.3f}')

In [ ]:
# ===== ויזואליזציה: נכונות ושגיאות =====

proba_cache = model.predict_proba(X_test)
pred_cache  = (proba_cache > 0.5).astype(int)
correct_mask = (pred_cache == y_test.astype(int))
error_mask   = ~correct_mask

correct_idx = np.where(correct_mask)[0]
error_idx   = np.where(error_mask)[0]
label_names = ['חתול', 'כלב']

def show_grid(indices, title, n=8):
    indices = indices[:n]
    fig, axes = plt.subplots(1, len(indices), figsize=(2*len(indices), 3))
    if len(indices) == 1: axes = [axes]
    for ax, idx in zip(axes, indices):
        img = np.clip(
            X_test[idx] * std.squeeze()[:,None,None] + mean.squeeze()[:,None,None],
            0, 1).transpose(1, 2, 0)
        ax.imshow(img)
        p = proba_cache[idx]
        true_l = label_names[int(y_test[idx])]
        pred_l = label_names[pred_cache[idx]]
        ax.set_title(f'{true_l}\n→{pred_l}\n{p:.2f}', fontsize=8)
        ax.axis('off')
    fig.suptitle(h(title), fontsize=12)
    plt.tight_layout()
    plt.show()

show_grid(np.random.choice(correct_idx, 8, replace=False), 'סיווגים נכונים')
show_grid(np.random.choice(error_idx,   8, replace=False), 'שגיאות')

In [ ]:
# ===== ווידג'ט אינטראקטיבי =====

out = widgets.Output()

def show_single(idx):
    img = np.clip(
        X_test[idx] * std.squeeze()[:,None,None] + mean.squeeze()[:,None,None],
        0, 1).transpose(1, 2, 0)
    p = float(proba_cache[idx])
    true_l = label_names[int(y_test[idx])]
    pred_l = label_names[int(p > 0.5)]
    correct = pred_l == true_l

    with out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(8, 4))
        axes[0].imshow(img)
        axes[0].set_title(
            h(f'אמת: {true_l} | ניבוי: {pred_l}') +
            ('  ✓' if correct else '  ✗'),
            color='green' if correct else 'red', fontsize=13)
        axes[0].axis('off')

        cats_p = 1 - p
        bars = axes[1].barh(['כלב', 'חתול'], [p, cats_p],
                             color=['royalblue', 'coral'])
        axes[1].set_xlim(0, 1)
        axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.5)
        axes[1].set_title('סיכויים', fontsize=12)
        for bar, v in zip(bars, [p, cats_p]):
            axes[1].text(v + 0.02, bar.get_y() + bar.get_height()/2,
                         f'{v:.3f}', va='center')
        plt.tight_layout()
        plt.show()

slider   = widgets.IntSlider(value=0, min=0, max=len(X_test)-1, description='דוגמה:', style={'description_width':'70px'})
btn_prev = widgets.Button(description='◀ הקודם')
btn_next = widgets.Button(description='הבא ▶')
btn_err  = widgets.Button(description='שגיאה הבאה ⚠️', button_style='warning')

err_ptr = [0]

def on_prev(_):  slider.value = max(0, slider.value - 1)
def on_next(_):  slider.value = min(len(X_test)-1, slider.value + 1)
def on_err(_):
    err_ptr[0] = (err_ptr[0] + 1) % len(error_idx)
    slider.value = error_idx[err_ptr[0]]

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_err.on_click(on_err)
widgets.interactive(lambda idx: show_single(idx), idx=slider)

display(widgets.VBox([
    widgets.HBox([btn_prev, slider, btn_next, btn_err]),
    out
]))
show_single(0)

<div dir=rtl style="text-align: right">

---

## סיכום ומסקנות

### מה בנינו

רשת CNN מלאה ב-NumPy בלבד:
- **2 שכבות Conv** עם ReLU ו-MaxPool
- **2 שכבות FC** עם Dropout
- **Adam optimizer** בכולן
- **Backprop מלא** — גזרנו כל גרדיאנט מהגדרה

### למה CNN עדיף על FC+HOG כאן?

| | FC + HOG | CNN |
|--|----------|-----|
| עיבוד מקדים | HOG ידני | אוטומטי |
| שיתוף משקלים | לא | כן |
| היררכיית פיצ'רים | קבוע | נלמד |
| דיוק (CIFAR cats/dogs) | ~78-82% | **~83-87%** |

### השלב הבא

שיפורים אפשריים:
1. **Data Augmentation** — flip אופקי, crop אקראי → +2-3%
2. **Batch Normalization** → אימון מהיר ויציב יותר
3. **Transfer Learning** — לקחת VGG/ResNet שאומן על ImageNet → 90%+
4. **יותר שכבות** — Conv3, Conv4...

### העיקרון המרכזי

> CNN = **locality** (פילטרים מקומיים) + **weight sharing** (אותו פילטר בכל מקום) + **hierarchy** (שכבות עמוקות)
> 
> אלו שלושת העקרונות שהפכו CNN לכוח השליט בcomputer vision עד עידן הטרנספורמרים.

</div>